# Retriever Evaluation

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from loader import load_data
from retrievers.bm25 import BM25Retriever
from retrievers.tf_idf_retriever import TFIDFRetriever
from retrievers.dense_retriever import DenseRetriever
from retrievers.hybrid_retriever import HybridRetriever
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time
import pandas as pd

In [ ]:
# Build corpus
ds = load_data(n=50_000)
passages_text = []
queries = []
for example in ds:
    queries.append(example["query"])
    for passage in example["passages"]["passage_text"]:
        passages_text.append(passage)

print(f"Total passages: {len(passages_text)}")
print(f"Total queries: {len(queries)}")

In [ ]:
import gc

# mrr_at_10 uses query_batch automatically when the retriever supports it.
# measure_retrieval_time also uses query_batch when available (capped at 200 queries).
MAX_QUERIES = 1_000_000
results = []

# BM25
bm25 = BM25Retriever(top_k=10)
bm25.fit(passages_text)
mrr = mrr_at_10(bm25, ds, max_queries=MAX_QUERIES)
avg_time = measure_retrieval_time(bm25, queries)
results.append({"Retriever": "BM25", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_time * 1000, 3)})
print(f"BM25 done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time * 1000:.3f}")
del bm25; gc.collect()

# TF-IDF
tfidf = TFIDFRetriever(top_k=10)
tfidf.fit(passages_text)
mrr = mrr_at_10(tfidf, ds, max_queries=MAX_QUERIES)
avg_time = measure_retrieval_time(tfidf, queries)
results.append({"Retriever": "TF-IDF", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_time * 1000, 3)})
print(f"TF-IDF done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time * 1000:.3f}")
del tfidf; gc.collect()

# Dense
dense = DenseRetriever(top_k=10)
dense.fit("sbert_embeddings.npy", passages_text)
mrr = mrr_at_10(dense, ds, max_queries=MAX_QUERIES)
avg_time = measure_retrieval_time(dense, queries)
results.append({"Retriever": "Dense", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_time * 1000, 3)})
print(f"Dense done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time * 1000:.3f}")
del dense; gc.collect()

# Hybrid
hybrid = HybridRetriever(top_k=10)
hybrid.fit(passages_text)
mrr = mrr_at_10(hybrid, ds, max_queries=MAX_QUERIES)
avg_time = measure_retrieval_time(hybrid, queries)
results.append({"Retriever": "Hybrid", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_time * 1000, 3)})
print(f"Hybrid done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time * 1000:.3f}")
del hybrid; gc.collect()

del passages_text, queries; gc.collect()

In [ ]:
# Display and save results
df = pd.DataFrame(results)
df.to_csv("evaluations.csv", index=False)
print(df.to_string(index=False))

## Test Retrievers Individually

In [ ]:
import gc

MAX_QUERIES = 1_000_000
ds = load_data(n=100_000)

passages_text = []
queries = []
for example in ds:
    queries.append(example["query"])
    for passage in example["passages"]["passage_text"]:
        passages_text.append(passage)

hybrid = HybridRetriever(top_k=10, candidate_k=100)
hybrid.fit(passages_text)
del passages_text; gc.collect()

mrr = mrr_at_10(hybrid, ds, max_queries=MAX_QUERIES)
avg_time = measure_retrieval_time(hybrid, queries)
print(f"Hybrid done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time * 1000:.3f}")
del hybrid, queries; gc.collect()